# FMAifs Rollout (Date-Range Controlled)

This notebook runs inference using the FMAifs model over a specified date range.

## What this does

- Loads preprocessed atmospheric states from `.pkl` files
- Runs autoregressive rollouts using a trained model
- Generates future states at fixed lead times
- Saves the resulting trajectory for each initialization time

## Key behavior

- Date-driven processing loop
- Missing files are skipped safely
- Outputs are saved per initialization time

In [ ]:
print("FMAifs ROLLOUT")

import torch
import pickle
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

In [ ]:
# -----------------------------
# CONFIG
# -----------------------------

# Assumptions:
# - Input files follow naming convention: YYYYMMDD_HH.pkl
# - Each file contains a dictionary with:
#     {
#         "date": datetime,
#         "fields": {key: np.ndarray}
#     }
# - Model predicts only a subset of fields
# - Time resolution:
#     - Input stepping: 12 hours
#     - Prediction lead time: 12 hours

input_path = Path("../era5/aifs_preprocessed")
output_path = Path("./HHO_AIFS_output")
output_path.mkdir(exist_ok=True)

checkpoint = "../checkpoints/aifs_single_v0.2.1.ckpt"

rollout_steps = 20

# define your date range here
start_date = datetime(2024, 4, 30, 12)
end_date   = datetime(2024, 5, 31, 0)

In [ ]:
# -----------------------------
# MODEL
# -----------------------------

# Assumptions:
# - CUDA is available
# - Checkpoint is compatible with SimpleRunner
# - Inference is run in eval mode (no gradients)

print("Loading model...")
runner = SimpleRunner(checkpoint, device="cuda")

## Main Processing Loop

This loop iterates over the specified date range and runs inference for each available input state.

Each iteration:
- Loads a state file
- Runs autoregressive rollouts
- Saves full trajectory output as a pickled file that will be regridded in postprocessing

In [ ]:
current_date = start_date

while current_date <= end_date:

    timestamp = current_date.strftime("%Y%m%d_%H")
    file = input_path / f"{timestamp}.pkl"

    print("\n====================================")
    print("Running:", file)
    print("====================================")

    if not file.exists():
        print("Missing file, skipping:", file)
        current_date += timedelta(hours=6)
        continue

    with open(file, "rb") as f:
        current_state = pickle.load(f)

    trajectory = [current_state]

    for step in range(rollout_steps):

        with torch.no_grad():
            next_state = list(runner.run(
                input_state=current_state,
                lead_time=12
            ))[-1]

        print(f"\n--- Step {step+1} ---")
        print_state(next_state)

        new_fields = {}
        predicted_keys = set(next_state["fields"].keys())

        for key in current_state["fields"]:

            if key not in predicted_keys:
                new_fields[key] = current_state["fields"][key]
                continue

            prev = current_state["fields"][key]
            new = next_state["fields"][key]

            if new.ndim == 1:
                new = new[None, :]

            new_fields[key] = np.concatenate([prev[1:2], new], axis=0)

        current_state = {
            "date": current_state["date"] + timedelta(hours=12),
            "fields": new_fields
        }

        trajectory.append(current_state)

    # -----------------------------
    # SAVE RAW TRAJECTORY
    # -----------------------------
    print("\nSaving raw trajectory...")

    out_file = output_path / f"HHO_aifs_rollout_{timestamp}.pkl"

    with open(out_file, "wb") as f:
        pickle.dump(trajectory, f)

    print("Saved:", out_file)

    current_date += timedelta(hours=12)
